In [1]:
# Multiple ways to get and work with home directory
import os
from pathlib import Path
import spacy
import scispacy
import pandas as pd
import math
import numpy as np
import requests
import re
import time
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sentence_transformers import SentenceTransformer, util
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity
from torch.nn.functional import cosine_similarity as torch_cosine_similarity
from functools import lru_cache
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import rankdata

In [179]:
# 1. Using os.path
home_dir = os.path.expanduser('~')
print(f"Home directory (os.path): {home_dir}")

# 2. Using Path from pathlib (modern approach)
home = Path.home()
print(f"Home directory (pathlib): {home}")

# 3. Using environment variables
home_env = os.environ['HOME']  # For Unix/Linux/Mac
# home_env = os.environ['USERPROFILE']  # For Windows
print(f"Home directory (environ): {home_env}")

# Example: List contents of home directory
#print("\nContents of home directory:")
#for item in Path.home().iterdir():
#    print(f"- {item.name}")

# Create a new folder (if it doesn't exist)
new_folder = "/Users/naveendeevi/Datasets"
if not os.path.exists(new_folder):
    os.makedirs(new_folder)

# Change directory to the new folder
os.chdir(new_folder)

# Verify current working directory
print(f"Current working directory: {os.getcwd()}")

Home directory (os.path): /Users/naveendeevi
Home directory (pathlib): /Users/naveendeevi
Home directory (environ): /Users/naveendeevi
Current working directory: /Users/naveendeevi/Datasets


### 1. Create a function for pulling the Citations & References from the OpenAlex API. 

In [182]:
def get_openalex_references_and_citations(work_id):
    """
    Given an OpenAlex Work ID, return a dictionary of its references and citations.

    Args:
        work_id (str): OpenAlex Work ID (e.g., "W4406696697")

    Returns:
        dict: {
            'id': str,
            'title': str,
            'referenced_works': List[str],
            'cited_by_works': List[str]
        }
    """
    base_url = f"https://api.openalex.org/works/{work_id}"
    response = requests.get(base_url)

    if response.status_code != 200:
        raise Exception(f"Error fetching data for {work_id}: {response.status_code}")

    data = response.json()
    referenced_works = data.get("referenced_works", [])
    
    # Get citing works via filter
    citations_url = f"https://api.openalex.org/works?filter=cites:{work_id}&per-page=200"
    citations_response = requests.get(citations_url)

    if citations_response.status_code != 200:
        raise Exception(f"Error fetching citations for {work_id}: {citations_response.status_code}")
    
    citations_data = citations_response.json()
    citing_ids = [item["id"].split("/")[-1] for item in citations_data.get("results", [])]

    return {
        "id": data.get("id", "").split("/")[-1],
        "title": data.get("title", ""),
        "referenced_works": referenced_works,
        "cited_by_works": citing_ids
    }


## Function that populates the Pool Collection DF. 
def create_openalex_entry_df(identifier):
    """
    Given an OpenAlex ID or link, return a DataFrame with metadata:
    ID, Title, API Link, Citation/Reference, and Author Names.

    Args:
        identifier (str): OpenAlex Work ID (e.g., 'W2046695653') or URL (e.g., 'https://openalex.org/W2049189688')

    Returns:
        pd.DataFrame: With columns ['ID', 'Title', 'API link', 'Citation or Reference', 'Authors']
    """
    # Determine if input is a link or ID
    if identifier.startswith("https://openalex.org/"):
        work_id = identifier.split("/")[-1]
        label = "Reference"
    else:
        work_id = identifier
        label = "Citation"

    # Build API link
    api_link = f"https://api.openalex.org/works/{work_id}"

    # Fetch paper metadata
    response = requests.get(api_link)
    if response.status_code != 200:
        raise Exception(f"Failed to retrieve OpenAlex data: {response.status_code}")

    data = response.json()

    title = data.get("title", "")
    authorships = data.get("authorships", [])

    # Extract author names
    author_names = [auth["author"]["display_name"] for auth in authorships if "author" in auth]
    authors_str = ", ".join(author_names) if author_names else ""

    # Assemble row
    row = {
        "ID": work_id,
        "Title": title,
        "API link": api_link,
        "Citation or Reference": label,
        "Authors": authors_str
    }

    return pd.DataFrame([row])

## Similarity scores SentenceBERT

# Corpus-wide SBERT Similarity (One-to-Many)
def sbert_group_similarity(df, ref_col, cmp_col, score_col='SBERT_group_similarity'):
    model = SentenceTransformer('all-MiniLM-L6-v2')

    df = df.copy()
    df['ref_group'] = (df[ref_col] != df[ref_col].shift()).cumsum()

    all_scores = {}

    for gid, group in df.groupby('ref_group'):
        ref_text = group[ref_col].iloc[0]

        # Skip entire group if reference text is missing or invalid
        if pd.isna(ref_text) or not isinstance(ref_text, str) or not ref_text.strip():
            for idx in group.index:
                all_scores[idx] = np.nan
            continue

        cmp_texts = group[cmp_col].tolist()
        cmp_indices = group.index.tolist()

        sim_scores = []
        for t in cmp_texts:
            if pd.isna(t) or not isinstance(t, str) or not t.strip():
                sim_scores.append(np.nan)
            else:
                emb_ref = model.encode(ref_text, convert_to_tensor=True)
                emb_cmp = model.encode(t, convert_to_tensor=True)
                score = util.cos_sim(emb_ref, emb_cmp).item()
                sim_scores.append(score)

        for i, idx in enumerate(cmp_indices):
            all_scores[idx] = sim_scores[i]

    df[score_col] = df.index.to_series().map(all_scores)
    return df.drop(columns=['ref_group'])

# Word cloud & Sentence similarity
def word_cloud_jaccard_similarity(word_list, sentence):
    """
    Compute Jaccard similarity between a list of words and a sentence.

    Args:
        word_list (list): A list of words (e.g., MDD_Word_Cloud)
        sentence (str): A sentence to compare with the word list

    Returns:
        float: Jaccard similarity score (0.0 to 1.0)
    """
    if not isinstance(sentence, str) or not sentence.strip():
        return 0.0

    # Convert both to sets
    set1 = set(w.lower() for w in word_list)
    set2 = set(sentence.lower().split())

    # Compute Jaccard index
    intersection = set1.intersection(set2)
    union = set1.union(set2)

    if not union:
        return 0.0

    return len(intersection) / len(union)

MDD_Word_Cloud = ["5-HT1A", "5-HT4R", "Agomelatine", "AMPA Receptor Activation", "AMPA antagonist", "Analgesics", "Anesthetics",
    "Antidepressant", "Antidepressants", "Antipsychotic", "Antipsychotics", "Ayahuasca", "BDI", "BDNF", "Beta-arrestin",
    "Biomarker", "Biomarkers", "Brexpiprazole", "Bupropion", "Citalopram", "Clomipramine", "CYP2C19", "CYP2D6",
    "Central Nervous System Depressants", "Cyclooxygenase Inhibitors", "Desvenlafaxine", "Dopamine Reuptake Inhibitor",
    "Depression", "Depress", "Depressive", "MDD", "Duloxetine", "EEG", "Escitalopram", "Estrogen Antagonists", 
    "Excitatory Amino Acid Agents", "Excitatory Amino Acid Antagonists",
    "Fluoxetine", "GABA Agents", "Gene", "GeneSight", "Genetic", "Genetics", "Genomics", "General Anesthetics",
    "Glutamate", "Glutamate antagonist", "HAMD", "HAM-D", "HDRS", "Hydroxynorketamine", "IGF-1", "Inventory (generic)",
    "Ketamine", "Leucine", "MADRS", "MDMA", "MTHFR", "Magnetic Resonance Spectroscopy", "Magnetoencephalography",
    "Metabotropic glutamate receptor antagonist", "Mirtazapine", "Montgomery-Åsberg Depression Rating Scale",
    "NEGR1", "NMDA antagonist", "NMDA receptor antagonist", "Norepinephrine Reuptake Inhibitor", "Opioid",
    "PACAP", "PHQ", "PHQ-8", "PHQ-9", "PRKCB1", "Paroxetine", "Patient Health Questionnaire", "Peripheral Nervous System Agents",
    "Psilocybin", "P2X7", "QEEG", "QIDS", "SERT", "SNRI", "SNRIs", "SSRI", "SSRIs", "SV2A", "Selective Serotonin Reuptake Inhibitor",
    "Sensory System Agents", "Serotonin Uptake Inhibitors", "Serotonin transporter gene polymorphism",
    "SPECT", "Trazodone", "Thyroid pathway genes", "Triiodothyronine", "Venlafaxine", "Vilazodone", "Vortioxetine"
]

# Create a function that takes a dataframe and give a Approved & Rejected dataframe
def take_pool_df_approved_rejected_df(pool_collection,similarity_threshold=0.10): 

    filter_sheet = pd.DataFrame(columns=["Seed_Title","reference_ID","reference_citation_title","API link","Authors","SBERT_score","word_cloud_score"])
    ###################################2. Pull citations & References from seed sheets #####################################
    for ids,title in zip(pool_collection['Seed_ID'],pool_collection['Seed_Title']):
        citations_references = get_openalex_references_and_citations(ids)
        #print(ids)
        #print('Seed title:%s'%title)
        #filter_sheet.loc[len(filter_sheet),'seed_title'] = title
        # Pull references
        for i in citations_references['referenced_works']:
            x = create_openalex_entry_df(i)
            #print('Title: %s'%x['Title'])
            #Sub_pool_collection = pd.concat([Sub_pool_collection,x])
            filter_sheet.loc[len(filter_sheet),['Seed_Title','reference_ID','reference_citation_title','API link','Authors']] = [title,x.loc[0,'ID'],x.loc[0,'Title'],x.loc[0,'API link'],x.loc[0,'Authors']]
        
        for j in citations_references['cited_by_works']:
            y = create_openalex_entry_df(j)
            #Sub_pool_collection = pd.concat([Sub_pool_collection,y])
            #filter_sheet.loc[len(filter_sheet),['seed_title','reference_citation_title']] = [title,y.loc[0,'Title']]
            filter_sheet.loc[len(filter_sheet),['Seed_Title','reference_ID','reference_citation_title','API link','Authors']] = [title,y.loc[0,'ID'],y.loc[0,'Title'],y.loc[0,'API link'],y.loc[0,'Authors']]

        
        ####################### 3. Add filtering criteria. a) SentenceBERT b)Word Cloud ##########################################
        filter_sheet_V2 = sbert_group_similarity(filter_sheet,'Seed_Title','reference_citation_title', score_col = 'SBERT_score')

        # Min-Max Normalization: scale values to range [0, 1]
        min_val = filter_sheet_V2['SBERT_score'].min()
        max_val = filter_sheet_V2['SBERT_score'].max()
        filter_sheet_V2['SBERT_score_normalized'] = (filter_sheet_V2['SBERT_score'] - min_val) / (max_val - min_val)
        filter_sheet_V2.drop(columns=['SBERT_score'],inplace=True)

        filter_sheet_V2['word_cloud_score'] = filter_sheet_V2['reference_citation_title'].apply(lambda sentence: word_cloud_jaccard_similarity(MDD_Word_Cloud, sentence))

        ########### UL 001 ###########################
        filter_sheet_V2['SBERT_percentile'] = rankdata(filter_sheet_V2['SBERT_score'], method='average') / len(filter_sheet_V2)
        filter_sheet_V2.drop(columns=['SBERT_score'], inplace=True)
        filter_sheet_V2['title_Status'] = np.where(
            (filter_sheet_V2['SBERT_percentile'] >= 0.95) | (filter_sheet_V2['word_cloud_score'] > 1.00),'collection_pool','rejection_pool')
        ########### End of UL 001 ######################
        
        
        # Filter for the papers that are above the similarity threshold and word cloud. 
        #filter_sheet_V2['title_Status'] = np.where((filter_sheet_V2['SBERT_score_normalized'] >= similarity_threshold) | (filter_sheet_V2['word_cloud_score'] > 0.00), 'collection_pool', 'rejection_pool')

    # === APPROVED POOL --> Pooled_collection ===
    approved_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'collection_pool']
    approved_df = approved_df[['Seed_Title','reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    approved_df.columns = ['Seed_Title','Linked_Paper_ID', 'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors']
    #approved_df['Citation or Reference'] = 'reference_&_citations'

    # === REJECTED POOL --> Rejection_pool ===
    rejected_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'rejection_pool']
    rejected_df = rejected_df[['reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    rejected_df.columns = ['Linked_Paper_ID', 'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors']
    #rejected_df['Citation or Reference'] = 'reference_&_citations'

    return approved_df,rejected_df

# Function that takes bigger 
def iteratable_approve_reject(jump_df,similarity_threshold = 0.10):
    filter_sheet = pd.DataFrame(columns=["Seed_Title","reference_ID","reference_citation_title","API link","Authors","SBERT_score","word_cloud_score"])
    ###################################2. Pull citations & References from seed sheets ##################################### 
    for ids,title,seed_title in zip(jump_df['Linked_Paper_ID'],jump_df['Linked_Paper_Title'],jump_df['Seed_Title']):
        citations_references = get_openalex_references_and_citations(ids)
        #print(ids)
        
        #print('Seed title:%s'%title)
        #filter_sheet.loc[len(filter_sheet),'seed_title'] = title
        # Pull references
        for i in citations_references['referenced_works']:
            x = create_openalex_entry_df(i)
            #print('Title: %s'%x['Title'])
            #Sub_pool_collection = pd.concat([Sub_pool_collection,x])
            filter_sheet.loc[len(filter_sheet),['Seed_Title','reference_ID','reference_citation_title','API link','Authors']] = [seed_title,x.loc[0,'ID'],x.loc[0,'Title'],x.loc[0,'API link'],x.loc[0,'Authors']]
        
        for j in citations_references['cited_by_works']:
            y = create_openalex_entry_df(j)
            #Sub_pool_collection = pd.concat([Sub_pool_collection,y])
            #filter_sheet.loc[len(filter_sheet),['seed_title','reference_citation_title']] = [title,y.loc[0,'Title']]
            filter_sheet.loc[len(filter_sheet),['Seed_Title','reference_ID','reference_citation_title','API link','Authors']] = [seed_title,y.loc[0,'ID'],y.loc[0,'Title'],y.loc[0,'API link'],y.loc[0,'Authors']]

        ####################### 3. Add filtering criteria. a) SentenceBERT b)Word Cloud ##########################################
        filter_sheet_V2 = sbert_group_similarity(filter_sheet,'Seed_Title','reference_citation_title', score_col = 'SBERT_score')

        # Min-Max Normalization: scale values to range [0, 1]
        min_val = filter_sheet_V2['SBERT_score'].min()
        max_val = filter_sheet_V2['SBERT_score'].max()
        filter_sheet_V2['SBERT_score_normalized'] = (filter_sheet_V2['SBERT_score'] - min_val) / (max_val - min_val)
        filter_sheet_V2.drop(columns=['SBERT_score'],inplace=True)

        filter_sheet_V2['word_cloud_score'] = filter_sheet_V2['reference_citation_title'].apply(lambda sentence: word_cloud_jaccard_similarity(MDD_Word_Cloud, sentence))

        ########### UL 001 ###########################
        filter_sheet_V2['SBERT_percentile'] = rankdata(filter_sheet_V2['SBERT_score'], method='average') / len(filter_sheet_V2)
        filter_sheet_V2.drop(columns=['SBERT_score'], inplace=True)
        filter_sheet_V2['title_Status'] = np.where(
            (filter_sheet_V2['SBERT_percentile'] >= 0.95) | (filter_sheet_V2['word_cloud_score'] > 1.00) ,'collection_pool','rejection_pool')
        ########### End of UL 001 ######################



        
        # Filter for the papers that are above the similarity threshold and word cloud. 
        #filter_sheet_V2['title_Status'] = np.where((filter_sheet_V2['SBERT_score_normalized'] >= similarity_threshold) | (filter_sheet_V2['word_cloud_score'] > 0.00), 'collection_pool', 'rejection_pool')

    # === APPROVED POOL --> Pooled_collection ===
    approved_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'collection_pool']
    approved_df = approved_df[['Seed_Title','reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    approved_df.columns = ['Seed_Title','Linked_Paper_ID', 'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors']
    #approved_df['Citation or Reference'] = 'reference_&_citations'

    # === REJECTED POOL --> Rejection_pool ===
    rejected_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'rejection_pool']
    rejected_df = rejected_df[['reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    rejected_df.columns = ['Linked_Paper_ID', 'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors']
    #rejected_df['Citation or Reference'] = 'reference_&_citations'

    return approved_df,rejected_df

import asyncio
import aiohttp
import nest_asyncio
import pandas as pd
from functools import lru_cache
import json
import os
import time
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from aiohttp import ClientTimeout

nest_asyncio.apply()

# === Caching Setup ===
CACHE_DIR = "openalex_cache"
EMBED_CACHE = os.path.join(CACHE_DIR, "embeddings")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(EMBED_CACHE, exist_ok=True)

# === Sentence-BERT Model ===
model = SentenceTransformer("all-MiniLM-L6-v2")

# === FAISS Index ===
embedding_dim = model.get_sentence_embedding_dimension()
faiss_index = faiss.IndexFlatL2(embedding_dim)
faiss_id_map = []

# === I/O Caching ===
def load_from_cache(work_id):
    path = os.path.join(CACHE_DIR, f"{work_id}.json")
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None

def save_to_cache(work_id, data):
    path = os.path.join(CACHE_DIR, f"{work_id}.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f)

def load_embedding_from_cache(text):
    path = os.path.join(EMBED_CACHE, f"{hash(text)}.npy")
    if os.path.exists(path):
        return np.load(path)
    return None

def save_embedding_to_cache(text, embedding):
    path = os.path.join(EMBED_CACHE, f"{hash(text)}.npy")
    np.save(path, embedding)

def cached_embedding(text):
    vec = load_embedding_from_cache(text)
    if vec is not None:
        return vec
    embedding = model.encode(text)
    save_embedding_to_cache(text, embedding)
    return embedding

# === Concurrency control ===
SEMAPHORE = asyncio.Semaphore(5)

# === OpenAlex Async Utilities ===
async def fetch_openalex_single_work(session, work_id):
    cached = load_from_cache(work_id)
    if cached:
        return cached
    url = f"https://api.openalex.org/works/{work_id}?mailto=naveen@headlamp.com"
    headers = {'User-Agent': 'HeadlampHealthBot/1.0 (mailto:naveen@headlamp.com)'}
    async with SEMAPHORE:
        async with session.get(url, headers=headers) as response:
            response.raise_for_status()
            data = await response.json()
            save_to_cache(work_id, data)
            return data

async def get_openalex_entries_async(id_list):
    timeout = ClientTimeout(total=30)
    async with aiohttp.ClientSession(timeout=timeout) as session:
        tasks = [fetch_openalex_single_work(session, work_id) for work_id in id_list]
        return pd.DataFrame(await asyncio.gather(*tasks))

# === FAISS Similarity ===
def index_embeddings(texts):
    global faiss_index, faiss_id_map
    vecs = [cached_embedding(text) for text in texts]
    faiss_index.add(np.vstack(vecs).astype('float32'))
    faiss_id_map.extend(texts)

def get_top_k_similar(query_text, k=5):
    q = cached_embedding(query_text).reshape(1, -1).astype('float32')
    D, I = faiss_index.search(q, k)
    return [(faiss_id_map[i], D[0][idx]) for idx, i in enumerate(I[0]) if i < len(faiss_id_map)]

# === Improved Approval Logic ===
def take_pool_df_approved_rejected_df(pool_collection, similarity_threshold=0.90):
    from collections import defaultdict

    all_ids = []
    seed_title_map = {}
    for _, row in pool_collection.iterrows():
        seed_id = row['Seed_ID']
        seed_title = row['Seed_Title']
        cached = load_from_cache(seed_id)
        if cached:
            refs = cached.get("referenced_works", []) + cached.get("cited_by_count", [])
            all_ids.extend(refs)
            for rid in refs:
                seed_title_map[rid] = seed_title

    all_ids = list(set(all_ids))
    cited_data = asyncio.get_event_loop().run_until_complete(get_openalex_entries_async(all_ids))

    titles = cited_data['title'].tolist()
    ids = [r['id'].split('/')[-1] for r in cited_data.to_dict('records')]
    titles_embeddings = [cached_embedding(title) for title in titles]

    approved = []
    for i, (title, eid, emb) in enumerate(zip(titles, ids, titles_embeddings)):
        seed_title = seed_title_map.get(eid, None)
        if not seed_title:
            continue
        seed_emb = cached_embedding(seed_title)
        sim = np.dot(seed_emb, emb) / (np.linalg.norm(seed_emb) * np.linalg.norm(emb))
        if sim >= similarity_threshold:
            approved.append({
                'Seed_Title': seed_title,
                'Linked_Paper_ID': eid,
                'Linked_Paper_Title': title,
                'Linked Paper API link': cited_data.iloc[i]['id'],
                'Linked Paper Authors': ", ".join([a['author']['display_name'] for a in cited_data.iloc[i]['authorships']])
            })

    return pd.DataFrame(approved), pd.DataFrame()


In [162]:

###########################1. Read the papers that Adam & Eleni collected. That is the seed paper. Add it to the seed paper ###############
#seed_papers = pd.read_csv('PHQ9_publication_index_full_titles_authors.csv')

seed_list = ['W4394693583','W1528826381']

#seed_list = ['W2148132316','W2114058973','W2148132316','W2114058973','W2153644230','W2742814779','W2338977821','W2014381681','W2012109041','W2140505555',
#             'W2775576435','W2257045848']

pool_collection = pd.DataFrame(columns=["Seed_ID", "Seed_Title", "Seed API link", "Seed Authors", "Seed or Citation or Reference"])


########################## 2. Pass the pool collection to a approval & rejection rubric to output Approval & Rejection list #############

#approved, rejected = take_pool_df_approved_rejected_df(pool_collection)

In [128]:
jump = 2
current_jump = 0

while current_jump <= jump: 
    if current_jump == 0:
        for i in seed_list:
            new_df = create_openalex_entry_df(i)
            mapped_df = new_df[['ID', 'Title', 'API link', 'Authors']].copy()
            mapped_df.columns = ['Seed_ID', 'Seed_Title', 'Seed API link', 'Seed Authors']
            mapped_df['Seed or Citation or Reference'] = 'Seed'
            pool_collection = pd.concat([pool_collection, mapped_df], ignore_index=True)
            print(pool_collection.shape)
    elif current_jump == 1:
        print("next iteration")
        approved, rejected = take_pool_df_approved_rejected_df(pool_collection,similarity_threshold=0.90)
        merged_df = pd.merge(pool_collection, approved, on='Seed_Title', how='inner')
        print(merged_df.shape)
    #print(pool_collection)
    elif current_jump > 1: 
        jump_df = merged_df.loc[:,['Seed_Title','Linked_Paper_ID','Linked_Paper_Title']]
        print(jump_df.shape)
        approved, rejected = iteratable_approve_reject(jump_df,similarity_threshold=0.90)
        merged_df_2 = pd.merge(pool_collection, approved, on='Seed_Title', how='inner')
        print(merged_df_2.shape)
        merged_df = pd.concat([merged_df,merged_df_2],ignore_index=True)
        print(merged_df.shape)
    current_jump+=1


(1, 5)
(2, 5)
next iteration


/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


(57, 9)
(57, 3)


/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be remove

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [ ]:
# 1. Merge results shape: 57, 9

In [101]:
# The first input is of shape: 2,5
# The first jump will give the shale as : merged_df.shape : 228,9


(228, 9)

In [145]:
jump = 2
current_jump = 0

pool_collection = pd.DataFrame(columns=["Seed_ID", "Seed_Title", "Seed API link", "Seed Authors", "Seed or Citation or Reference"])
merged_df = pd.DataFrame()

while current_jump <= jump & current_jump < 8:
    if current_jump == 0:
        for i in seed_list:
            new_df = create_openalex_entry_df(i)
            mapped_df = new_df[['ID', 'Title', 'API link', 'Authors']].copy()
            mapped_df.columns = ['Seed_ID', 'Seed_Title', 'Seed API link', 'Seed Authors']
            mapped_df['Seed or Citation or Reference'] = 'Seed'
            pool_collection = pd.concat([pool_collection, mapped_df], ignore_index=True)
        print("Jump 0 - pool_collection:", pool_collection.shape)

    elif current_jump == 1:
        print("Jump 1 - seed expansion")
        assert {'Seed_ID', 'Seed_Title'}.issubset(pool_collection.columns), "pool_collection missing required columns"

        approved, rejected = take_pool_df_approved_rejected_df(pool_collection, similarity_threshold=0.90)
        merged_df = pd.merge(pool_collection, approved, on='Seed_Title', how='inner')

        merged_df['Seed_Anchor_ID'] = merged_df['Seed_ID']
        merged_df['Seed_Anchor_Title'] = merged_df['Seed_Title']
        merged_df['Jump_Level'] = 1

        print("Jump 1 - merged_df:", merged_df.shape)

    elif current_jump > 1:
        print(f"Jump {current_jump} - recursive expansion")

        assert {'Linked_Paper_ID', 'Linked_Paper_Title', 'Seed_Anchor_ID', 'Seed_Anchor_Title'}.issubset(merged_df.columns), "merged_df missing required columns for jump > 1"

        # Prepare jump_df with correct structure expected by iteratable_approve_reject
        jump_df = merged_df[['Linked_Paper_ID', 'Linked_Paper_Title', 'Seed_Anchor_ID', 'Seed_Anchor_Title']].drop_duplicates().copy()
        jump_df.columns = ['Seed_ID', 'Seed_Title', 'Seed_Anchor_ID', 'Seed_Anchor_Title']

        print("Input to iteratable_approve_reject:", jump_df.columns.tolist())
        approved, rejected = iteratable_approve_reject_V2(jump_df, similarity_threshold=0.90)

        merged_df_2 = pd.merge(jump_df, approved, on='Seed_Title', how='inner')
        merged_df_2['Jump_Level'] = current_jump

        if 'Linked_Paper_ID' in merged_df.columns:
            merged_df_2 = merged_df_2[~merged_df_2['Linked_Paper_ID'].isin(merged_df['Linked_Paper_ID'])]

        merged_df = pd.concat([merged_df, merged_df_2], ignore_index=True)
        print(f"Jump {current_jump} - total merged_df so far:", merged_df.shape)

    current_jump += 1

Jump 0 - pool_collection: (2, 5)


In [183]:
import asyncio
import aiohttp
import nest_asyncio
import pandas as pd
import json
import os
import time

nest_asyncio.apply()

# === Caching Setup ===
CACHE_DIR = "openalex_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# === OpenAlex Utilities ===
def normalize_id(work_id):
    return work_id.split('/')[-1]

def load_from_cache(work_id):
    work_id = normalize_id(work_id)
    path = os.path.join(CACHE_DIR, f"{work_id}.json")
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None

def save_to_cache(work_id, data):
    work_id = normalize_id(work_id)
    path = os.path.join(CACHE_DIR, f"{work_id}.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f)

async def fetch_openalex_single_work(session, work_id, retries=3, delay=2):
    cached = load_from_cache(work_id)
    if cached:
        return cached

    url = f"https://api.openalex.org/works/{normalize_id(work_id)}?mailto=naveen@headlamp.com"
    headers = {'User-Agent': 'HeadlampHealthBot/1.0 (mailto:naveen@headlamp.com)'}

    for attempt in range(retries):
        try:
            async with session.get(url, headers=headers) as response:
                if response.status == 429:
                    await asyncio.sleep(delay * (2 ** attempt))
                    continue
                response.raise_for_status()
                data = await response.json()
                save_to_cache(work_id, data)
                return data
        except aiohttp.ClientError:
            await asyncio.sleep(delay * (2 ** attempt))
            continue
    raise Exception(f"Failed after {retries} retries for {work_id}")

async def get_openalex_entries_async(id_list):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_openalex_single_work(session, work_id) for work_id in id_list]
        return pd.DataFrame(await asyncio.gather(*tasks))

def take_pool_df_approved_rejected_df(pool_collection, similarity_threshold=0.90):
    filter_sheet = pd.DataFrame(columns=["Seed_Title","reference_ID","reference_citation_title","API link","Authors","SBERT_score","word_cloud_score"])

    for ids,title in zip(pool_collection['Seed_ID'],pool_collection['Seed_Title']):
        citations_references = get_openalex_references_and_citations(ids)
        for i in citations_references['referenced_works']:
            x = create_openalex_entry_df(i)
            filter_sheet.loc[len(filter_sheet),['Seed_Title','reference_ID','reference_citation_title','API link','Authors']] = [title,x.loc[0,'ID'],x.loc[0,'Title'],x.loc[0,'API link'],x.loc[0,'Authors']]
        for j in citations_references['cited_by_works']:
            y = create_openalex_entry_df(j)
            filter_sheet.loc[len(filter_sheet),['Seed_Title','reference_ID','reference_citation_title','API link','Authors']] = [title,y.loc[0,'ID'],y.loc[0,'Title'],y.loc[0,'API link'],y.loc[0,'Authors']]

    filter_sheet_V2 = sbert_group_similarity(filter_sheet,'Seed_Title','reference_citation_title', score_col = 'SBERT_score')
    min_val = filter_sheet_V2['SBERT_score'].min()
    max_val = filter_sheet_V2['SBERT_score'].max()
    filter_sheet_V2['SBERT_score_normalized'] = (filter_sheet_V2['SBERT_score'] - min_val) / (max_val - min_val)
    filter_sheet_V2.drop(columns=['SBERT_score'],inplace=True)
    filter_sheet_V2['word_cloud_score'] = filter_sheet_V2['reference_citation_title'].apply(lambda sentence: word_cloud_jaccard_similarity(MDD_Word_Cloud, sentence))
    filter_sheet_V2['title_Status'] = np.where((filter_sheet_V2['SBERT_score_normalized'] >= similarity_threshold) | (filter_sheet_V2['word_cloud_score'] > 0.00), 'collection_pool', 'rejection_pool')

    approved_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'collection_pool']
    approved_df = approved_df[['Seed_Title','reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    approved_df.columns = ['Seed_Title','Linked_Paper_ID', 'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors']

    rejected_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'rejection_pool']
    rejected_df = rejected_df[['reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    rejected_df.columns = ['Linked_Paper_ID', 'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors']

    return approved_df,rejected_df

def iteratable_approve_reject(jump_df, similarity_threshold=0.90):
    filter_sheet = pd.DataFrame(columns=["Seed_Title","reference_ID","reference_citation_title","API link","Authors","SBERT_score","word_cloud_score"])

    for ids, title, seed_title in zip(jump_df['Seed_ID'], jump_df['Seed_Title'], jump_df['Seed_Anchor_Title']):
        citations_references = get_openalex_references_and_citations(ids)
        for i in citations_references['referenced_works']:
            x = create_openalex_entry_df(i)
            filter_sheet.loc[len(filter_sheet),['Seed_Title','reference_ID','reference_citation_title','API link','Authors']] = [seed_title,x.loc[0,'ID'],x.loc[0,'Title'],x.loc[0,'API link'],x.loc[0,'Authors']]
        for j in citations_references['cited_by_works']:
            y = create_openalex_entry_df(j)
            filter_sheet.loc[len(filter_sheet),['Seed_Title','reference_ID','reference_citation_title','API link','Authors']] = [seed_title,y.loc[0,'ID'],y.loc[0,'Title'],y.loc[0,'API link'],y.loc[0,'Authors']]

    filter_sheet_V2 = sbert_group_similarity(filter_sheet,'Seed_Title','reference_citation_title', score_col = 'SBERT_score')
    min_val = filter_sheet_V2['SBERT_score'].min()
    max_val = filter_sheet_V2['SBERT_score'].max()
    filter_sheet_V2['SBERT_score_normalized'] = (filter_sheet_V2['SBERT_score'] - min_val) / (max_val - min_val)
    filter_sheet_V2.drop(columns=['SBERT_score'], inplace=True)
    filter_sheet_V2['word_cloud_score'] = filter_sheet_V2['reference_citation_title'].apply(lambda sentence: word_cloud_jaccard_similarity(MDD_Word_Cloud, sentence))
    filter_sheet_V2['title_Status'] = np.where((filter_sheet_V2['SBERT_score_normalized'] >= similarity_threshold) | (filter_sheet_V2['word_cloud_score'] > 0.00), 'collection_pool', 'rejection_pool')

    approved_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'collection_pool']
    approved_df = approved_df[['Seed_Title','reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    approved_df.columns = ['Seed_Title','Linked_Paper_ID', 'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors']

    rejected_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'rejection_pool']
    rejected_df = rejected_df[['reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    rejected_df.columns = ['Linked_Paper_ID', 'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors']

    return approved_df, rejected_df

# === Main Expansion ===
jump = 2
current_jump = 0
seed_list = ['W4394693583', 'W1528826381']

pool_collection = pd.DataFrame(columns=["Seed_ID", "Seed_Title", "Seed API link", "Seed Authors", "Seed or Citation or Reference"])
merged_df = pd.DataFrame()

while current_jump <= jump:
    start_time = time.time()

    if current_jump == 0:
        print(f"\nJump {current_jump} - Initializing seed metadata")
        for i in seed_list:
            new_df = asyncio.get_event_loop().run_until_complete(get_openalex_entries_async([i]))
            row = new_df.iloc[0]
            pool_collection = pd.concat([pool_collection, pd.DataFrame([{
                'Seed_ID': row['id'].split('/')[-1],
                'Seed_Title': row['title'],
                'Seed API link': row['id'],
                'Seed Authors': ", ".join([a['author']['display_name'] for a in row['authorships']]),
                'Seed or Citation or Reference': 'Seed'
            }])], ignore_index=True)
        print(f"Jump {current_jump} - pool_collection: {pool_collection.shape}")

    elif current_jump == 1:
        print(f"\nJump {current_jump} - First hop expansion")
        approved, rejected = take_pool_df_approved_rejected_df(pool_collection, similarity_threshold=0.50)
        merged_df = pd.merge(pool_collection, approved, on='Seed_Title', how='inner')
        merged_df['Seed_Anchor_ID'] = merged_df['Seed_ID']
        merged_df['Seed_Anchor_Title'] = merged_df['Seed_Title']
        merged_df['Jump_Level'] = 1
        print(f"Jump {current_jump} - merged_df: {merged_df.shape}")

    elif current_jump > 1:
        print(f"\nJump {current_jump} - Recursive expansion")

        if merged_df.empty:
            print("No merged_df from previous step. Skipping.")
            break

        jump_df = merged_df[['Seed_Anchor_ID', 'Seed_Anchor_Title', 'Linked_Paper_ID', 'Linked_Paper_Title']]\
                    .drop_duplicates().copy()
        jump_df.columns = ['Seed_Anchor_ID', 'Seed_Anchor_Title', 'Seed_ID', 'Seed_Title']

        approved, rejected = iteratable_approve_reject(jump_df, similarity_threshold=0.50)

        if approved.empty:
            print(f"No new approved papers at jump {current_jump}")
            break

        merged_df_2 = pd.merge(jump_df, approved, on='Seed_Title', how='inner')
        merged_df_2['Jump_Level'] = current_jump

        if 'Linked_Paper_ID' in merged_df.columns:
            merged_df_2 = merged_df_2[~merged_df_2['Linked_Paper_ID'].isin(merged_df['Linked_Paper_ID'])]

        merged_df = pd.concat([merged_df, merged_df_2], ignore_index=True)
        print(f"Jump {current_jump} - merged_df updated: {merged_df.shape}")

    total_time = time.time() - start_time
    print(f"Jump {current_jump} - TOTAL time: {total_time:.2f}s")

    current_jump += 1



Jump 0 - Initializing seed metadata
Jump 0 - pool_collection: (2, 5)
Jump 0 - TOTAL time: 0.00s

Jump 1 - First hop expansion


/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Jump 1 - merged_df: (115, 12)
Jump 1 - TOTAL time: 148.14s

Jump 2 - Recursive expansion


KeyboardInterrupt: 

In [89]:
pool_collection_dup = merged_df.loc[0:1,['Seed_Title','Linked_Paper_ID','Linked_Paper_Title']]

In [90]:
pool_collection_dup

,Seed_Title,Linked_Paper_ID,Linked_Paper_Title
0,Acute Adverse Effects of Therapeutic Doses of ...,W2006949123,Possible involvement of the central dopaminerg...
1,Acute Adverse Effects of Therapeutic Doses of ...,W2018241451,Identification of serotonin 5-HT1A receptor pa...


In [91]:
approved_df,rejected_df = iteratable_approve_reject(pool_collection_dup)

/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [93]:
# First row has 67 approved, 2 rejected
approved_df

,Seed_Title,Linked_Paper_ID,Linked_Paper_Title,Linked Paper API link,Linked Paper Authors
0,Acute Adverse Effects of Therapeutic Doses of ...,W1881954819,AN EXPERIMENTAL STUDY OF THE PARTICIPATION OF ...,https://api.openalex.org/works/W1881954819,"John T. Elder, James M. Dille"
1,Acute Adverse Effects of Therapeutic Doses of ...,W1967574354,The influence of quipazine on the turnover rat...,https://api.openalex.org/works/W1967574354,"M Grabowska, L Antkiewicz, Jerzy Michaluk"
2,Acute Adverse Effects of Therapeutic Doses of ...,W1976195294,DRUGS WHICH ANTAGONIZE 5‐HYDROXYTRYPTAMINE,https://api.openalex.org/works/W1976195294,"J. H. Gaddum, Khan A. Hameed"
3,Acute Adverse Effects of Therapeutic Doses of ...,W1981456480,Possible Interrelationship between Release of ...,https://api.openalex.org/works/W1981456480,"Bernard B. Brodie, Jacqueline S. Olin, R. Kunt..."
4,Acute Adverse Effects of Therapeutic Doses of ...,W1983288641,Single Extraction Method for the Simultaneous ...,https://api.openalex.org/works/W1983288641,"R. M. Fleming, William G. Clark"
...,...,...,...,...,...
139,Acute Adverse Effects of Therapeutic Doses of ...,W2464399287,"The Effects of Curcuma domestica, Zingiber off...",https://api.openalex.org/works/W2464399287,Brett Martin
140,Acute Adverse Effects of Therapeutic Doses of ...,W2475285632,Morphological Effects of Hydroalcoholic Zingib...,https://api.openalex.org/works/W2475285632,"Fariba Ghodrati, Minoo Mahmoodi, ‬Siamak Shahi..."
142,Acute Adverse Effects of Therapeutic Doses of ...,W2267566944,In Vitro Immunopharmacological Profiling of Gi...,https://api.openalex.org/works/W2267566944,Andreas Nievergelt
145,Acute Adverse Effects of Therapeutic Doses of ...,W4386321011,Pharmacognostic Evaluation of Zingiber Officin...,https://api.openalex.org/works/W4386321011,"Anwardi Jamil, Lim Pei San, Nik Muhammad Azhar..."


In [53]:
merged_df.columns

Index(['Seed_ID', 'Seed_Title', 'Seed API link', 'Seed Authors',
       'Seed or Citation or Reference', 'Linked_Paper_ID',
       'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors'],
      dtype='object')

In [7]:
pool_collection

,Seed_ID,Seed_Title,Seed API link,Seed Authors,Seed or Citation or Reference
0,W4394693583,Acute Adverse Effects of Therapeutic Doses of ...,https://api.openalex.org/works/W4394693583,"Akhila Yerubandi, Jennifer E. Thomas, N. M. Ma...",Seed
1,W4394693583,Acute Adverse Effects of Therapeutic Doses of ...,https://api.openalex.org/works/W4394693583,"Akhila Yerubandi, Jennifer E. Thomas, N. M. Ma...",Seed


In [12]:
approved.head(3)

,seed_title,Linked_Paper_ID,Linked_Paper_Title,Linked Paper API link,Linked Paper Authors
0,Acute Adverse Effects of Therapeutic Doses of ...,W2006949123,Possible involvement of the central dopaminerg...,https://api.openalex.org/works/W2006949123,"M.K. Menon, W. G. Clark, David T. Masuoka"
1,Acute Adverse Effects of Therapeutic Doses of ...,W2018241451,Identification of serotonin 5-HT1A receptor pa...,https://api.openalex.org/works/W2018241451,"Andreas Nievergelt, Peter Huonker, Roland Scho..."
2,Acute Adverse Effects of Therapeutic Doses of ...,W2022443784,"Psilocybin dose-dependently causes delayed, tr...",https://api.openalex.org/works/W2022443784,"Matthew W. Johnson, R. Andrew Sewell, Roland R..."
